# สัปดาห์ที่ 2 — แล็บ: ข้อมูลเชิงพื้นที่และ CRS (Vector · Raster · Data Cube)

**วิชา:** ปัญญาประดิษฐ์เชิงพื้นที่ (GeoAI) · สาขาภูมิสารสนเทศศาสตร์ มข.
**รูปแบบ:** ทำตามกล่อง "✅ ทำตอนนี้" ของแต่ละส่วน (ใช้ได้ทั้งแบบมีอาจารย์ในห้องและเรียนด้วยตนเอง) ใช้เวลาประมาณ 3 ชั่วโมง · **CLO ที่เกี่ยวข้อง:** CLO2 (เตรียมข้อมูลเชิงพื้นที่), CLO4 (Vibe Coding + Explain This)

## เป้าหมายของแล็บนี้
1. อ่าน "บัตรประจำตัว" ของชั้นข้อมูลเวกเตอร์และราสเตอร์ให้เป็น: **CRS · หน่วย · ความละเอียด · ขอบเขต**
2. ทำ **reproject · resample · clip · rasterize** กับข้อมูลขอนแก่นได้ และเลือกวิธี resample ให้ตรงชนิดข้อมูล
3. จับบั๊กเชิงพื้นที่ที่ "รันผ่านแต่ผิด" ในโค้ดที่ AI เขียน (Bug Hunt)
4. เห็นว่า data cube (เวลา × แบนด์ × y × x) คือ array หลายมิติที่ติดป้ายชื่อมิติด้วย Xarray
5. เขียน prompt ที่ **ระบุ CRS ชัดเจน** แล้วตรวจผลด้วย Verify on map

## แผนเวลาที่แนะนำ
| เวลาสะสม | กิจกรรม |
|---|---|
| 0:00–0:20 | ส่วน 0–1: ตั้งค่า, ข้อมูลนักศึกษา, Prompt Log, ตัวช่วย `check_layer` |
| 0:20–0:45 | ส่วน 2–3: โหลดข้อมูล + เวกเตอร์ + องศากับเมตร |
| 0:45–1:15 | ส่วน 4: ราสเตอร์ + reproject/resample |
| 1:15–1:25 | พัก |
| 1:25–2:05 | ส่วน 5: Bug Hunt (4 บั๊ก) |
| 2:05–2:35 | ส่วน 6: clip + rasterize |
| 2:35–2:50 | ส่วน 7: data cube |
| 2:50–3:10 | ส่วน 8–9: Mission 2, Verify on map, Prompt Log, ส่งงาน |

> **กติกาตลอดแล็บ (เหมือนสัปดาห์ 1):** ก่อนรันเซลล์ที่มีป้าย 📝 ให้เขียน *Explain This* ด้วยคำของตัวเอง · ทุกครั้งที่ใช้ AI ให้ `log_prompt(...)` · ผลลัพธ์ที่ตัวเลขสวยไม่พอ ต้อง **ดูบนแผนที่**

## 🧭 แผนที่นำทาง — อ่านก่อนเริ่ม (1 นาที)

**ก่อนทำอะไร:** เปิด notebook จาก Google Classroom (ระบบทำสำเนาให้แล้ว) หรือถ้าเปิดจากลิงก์ที่ใครแชร์ ให้ **File → Save a copy in Drive** ก่อน

| สัญลักษณ์ | ความหมาย |
|---|---|
| ▶️ | รันเซลล์นี้เลย ไม่ต้องแก้อะไร |
| ✏️ | แก้ค่าหรือกรอกข้อมูลของคุณก่อนรัน |
| 🤖 | ให้ AI ช่วย แล้วต้อง `log_prompt(...)` |
| 📝 | เขียน *Explain This* หรือคำตอบด้วยคำของคุณเอง |
| 👀 | ตรวจบนภาพ/แผนที่ ไม่เชื่อตัวเลขอย่างเดียว |

| ส่วน | ทำอะไร | ผลที่ควรเห็น | เสร็จ |
|---|---|---|---|
| 0 | ตั้งค่า + ข้อมูลนักศึกษา | เวอร์ชันไลบรารีครบ | ☐ |
| 1 | Prompt Log + `check_layer` (Explain This #1) | ไม่มีข้อความออก (ปกติ) | ☐ |
| 2 | โหลดข้อมูลขอนแก่น 3 ชั้น | ตาราง "ที่มาของข้อมูล" | ☐ |
| 3 | เวกเตอร์: CRS, องศากับเมตร, พื้นที่ (Explain This #2) | พื้นที่อำเภอ กม.² ที่ถูกต้อง | ☐ |
| 4 | ราสเตอร์ + reproject/resample (Explain This #3, #4) | ตารางเทียบ 3 ชั้น + ภาพเทียบ nearest/bilinear | ☐ |
| 5 | **Bug Hunt** 4 บั๊ก (Explain This #5) | ชี้บรรทัดที่ผิด + แก้แล้วผลถูก | ☐ |
| 6 | clip + rasterize (Explain This #6) | แผนที่ NDVI ตัดตามอำเภอ + NDVI เฉลี่ยรายอำเภอ | ☐ |
| 7 | data cube (Explain This #7) | กราฟ NDVI ตามเวลา | ☐ |
| 8 | Mission 2 + Verify on map | แผนที่ซ้อนชั้น + เช็กลิสต์ CRS | ☐ |
| 9 | สะท้อนคิด + ส่งงาน | `บันทึกไฟล์: prompt_log_week02.md` | ☐ |

**ถ้าเจอปัญหา**

| อาการ | ทำอย่างไร |
|---|---|
| `NameError: name '...' is not defined` | ยังไม่ได้รันเซลล์ก่อนหน้า → Runtime → Run before (หรือรันจากบนลงล่าง) |
| ดึงข้อมูลจริงไม่ได้ ขึ้น ⚠️ "ใช้ชุดจำลอง" | ทำงานต่อได้ (โครงสร้างเหมือนของจริง) แต่ **ตัวเลขและแผนที่ไม่ใช่ของจริง** — บันทึกไว้ใน Prompt Log และแจ้งอาจารย์ |
| ค้างที่ดาวน์โหลดนานกว่า 3 นาที | Runtime → Interrupt execution แล้วรันเซลล์นั้นใหม่ |

## ส่วน 0 — ตั้งค่า Colab

> ### ✅ ทำตอนนี้ — ส่วน 0 (≈ 5 นาที)
> 1. ▶️ รันเซลล์ **0.1** (ติดตั้ง `rioxarray` และ `pystac-client` ใช้เวลา 1 นาทีโดยประมาณ)
> 2. ▶️ รันเซลล์ **0.2** → ควรเห็นเวอร์ชันของไลบรารีครบทุกตัว
> 3. ✏️ แก้เซลล์ข้อมูลนักศึกษา 3 บรรทัด แล้วรัน

In [ ]:
# 0.1 ติดตั้งไลบรารีที่ Colab ยังไม่มี (รันครั้งเดียว)
!pip install -q rioxarray pystac-client

In [ ]:
# 0.2 ตรวจสภาพแวดล้อม
import sys, warnings
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr
import rioxarray, rasterio, pyproj, shapely, matplotlib
import matplotlib.pyplot as plt
print("Python:", sys.version.split()[0])
for m in (gpd, xr, rioxarray, rasterio, pyproj, shapely):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
STUDENT_ID   = "6XXXXXXX-X"          # <- แก้เป็นรหัสนักศึกษา
STUDENT_NAME = "ชื่อ นามสกุล"        # <- แก้เป็นชื่อของคุณ
AI_TOOL      = "ระบุชื่อ AI ที่ใช้ช่วยเขียนโค้ด เช่น Gemini ใน Colab / Claude / ChatGPT"

## ส่วน 1 — ตัวช่วย: Prompt Log และ `check_layer`

สองฟังก์ชันนี้อาจารย์เขียนไว้ให้ ใช้ตลอดแล็บ — `log_prompt` เหมือนสัปดาห์ 1 ส่วน `check_layer` คือ **"บัตรประจำตัวชั้นข้อมูล"** ที่คุณจะเรียกทุกครั้งก่อนเอาข้อมูลสองชั้นมาซ้อนกัน

> ### ✅ ทำตอนนี้ — ส่วน 1 (≈ 15 นาที)
> 1. ▶️ รันเซลล์ `log_prompt` และเซลล์ `check_layer`
> 2. 📝 อ่านโค้ด `check_layer` ทีละบรรทัด แล้วเขียน **Explain This #1** ใต้เซลล์ (ต้องบอกให้ได้ว่าทำไมต้องแยกเวกเตอร์กับราสเตอร์ และ "หน่วย" มาจากไหน)
> 3. ▶️ ลองเรียก `check_layer` กับข้อมูลง่าย ๆ ในเซลล์ทดสอบ แล้วอ่านผล

In [ ]:
from datetime import datetime

PROMPT_LOG = []

def log_prompt(prompt, result, verdict, note=""):
    """บันทึก 1 รอบของการคุยกับ AI · verdict: "ผ่าน" | "แก้" | "ทิ้ง" """
    assert verdict in ("ผ่าน", "แก้", "ทิ้ง"), "verdict ต้องเป็น ผ่าน / แก้ / ทิ้ง"
    PROMPT_LOG.append(dict(time=datetime.now().strftime("%H:%M"), prompt=prompt.strip(),
                           result=result.strip(), verdict=verdict, note=note.strip()))
    print(f"บันทึก Prompt #{len(PROMPT_LOG)} แล้ว ({verdict})")

def _cell(s):
    return s.replace("|", "\\|").replace("\n", "<br>")

def save_prompt_log(path="prompt_log_week02.md"):
    lines = [f"# Prompt Log — สัปดาห์ 2 — {STUDENT_NAME} ({STUDENT_ID})", "",
             f"เครื่องมือ AI: {AI_TOOL}", "",
             "| # | เวลา | Prompt | ผลที่ได้ | Review | หมายเหตุ |", "|---|---|---|---|---|---|"]
    for i, r in enumerate(PROMPT_LOG, 1):
        lines.append(f"| {i} | {r['time']} | {_cell(r['prompt'])} | {_cell(r['result'])} | {r['verdict']} | {_cell(r['note'])} |")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("บันทึกไฟล์:", path, "| จำนวน prompt:", len(PROMPT_LOG))

In [ ]:
LAYER_ROWS = []   # เก็บ "บัตรประจำตัว" ของทุกชั้นที่ตรวจ เพื่อเทียบกันเป็นตาราง

def check_layer(obj, name="layer", quiet=False):
    """พิมพ์ 'บัตรประจำตัว' ของชั้นข้อมูล: ชนิด · CRS · หน่วย · ความละเอียด · ขอบเขต · ขนาด"""
    if isinstance(obj, gpd.GeoDataFrame):
        kind, crs = "vector", obj.crs
        res = "—"
        bounds = tuple(float(v) for v in obj.total_bounds)
        size = f"{len(obj)} features ({', '.join(sorted(obj.geom_type.unique()))})"
    else:
        kind, crs = "raster", obj.rio.crs
        res = tuple(round(abs(float(r)), 6) for r in obj.rio.resolution())
        bounds = tuple(float(v) for v in obj.rio.bounds())
        size = f"shape={tuple(obj.shape)} dtype={obj.dtype}"
    crs = None if crs is None else pyproj.CRS.from_user_input(crs)      # ทำให้เป็นชนิดเดียวกันทั้งเวกเตอร์และราสเตอร์
    unit = "ไม่มี CRS!" if crs is None else crs.axis_info[0].unit_name
    row = dict(layer=name, kind=kind, crs=(crs.to_string() if crs else None), unit=unit, res=res,
               bounds=tuple(round(b, 4) for b in bounds), size=size)
    LAYER_ROWS.append(row)
    if not quiet:
        print(f"[{name}] {kind} | CRS: {row['crs']} | หน่วย: {unit} | ความละเอียด: {res}")
        print(f"   ขอบเขต (ซ้าย, ล่าง, ขวา, บน): {row['bounds']}")
        print(f"   ขนาด: {size}")
    return row

### Explain This #1 — `check_layer`
- **ทำอะไร:** _(เขียนด้วยคำของคุณเอง)_
- **ทำไมต้องแยกกรณี GeoDataFrame กับ raster:** _..._
- **จุดเสี่ยง + วิธีตรวจ:** _(เช่น ถ้า `crs` เป็น None จะเกิดอะไรกับการซ้อนชั้น)_

In [ ]:
# เซลล์ทดสอบ: ลองส่งข้อมูลเล็ก ๆ เข้า check_layer
test_pts = gpd.GeoDataFrame({"name": ["ทดสอบ"]}, geometry=gpd.points_from_xy([102.83], [16.43]), crs="EPSG:4326")
_ = check_layer(test_pts, "จุดทดสอบ")
LAYER_ROWS.clear()   # ล้างแถวทดสอบ ก่อนเริ่มตรวจชั้นข้อมูลจริง

## ส่วน 2 — โหลดข้อมูลขอนแก่น 3 ชั้น (โค้ดอ้างอิงจากอาจารย์)

เราจะใช้ข้อมูลสามชั้น ซึ่ง **ตั้งใจให้อยู่คนละ CRS และคนละความละเอียด**:

| ชั้น | ชนิด | แหล่ง | CRS ที่คาด |
|---|---|---|---|
| ขอบเขตอำเภอ | เวกเตอร์ (polygon) | geoBoundaries (เปิดสาธารณะ) | EPSG:4326 (องศา) |
| ภาพ Sentinel-2 (3 วัน, 4 แบนด์: blue green red nir) | ราสเตอร์ → data cube | Earth Search STAC | UTM 48N (เมตร, 10 ม.) |
| ความหนาแน่นประชากร WorldPop 2020 | ราสเตอร์ | WorldPop | EPSG:4326 (องศา, ~1 กม.) |

> **ถ้าดึงข้อมูลจริงไม่ได้** เซลล์จะสลับไปใช้ "ชุดข้อมูลจำลอง" ที่โครงสร้างเหมือนกัน และพิมพ์ ⚠️ ชัดเจน — โค้ดและแนวคิดทำต่อได้ แต่ตัวเลข/แผนที่ **ไม่ใช่ของจริง** ห้ามนำไปสรุปผลเชิงพื้นที่

> ### ✅ ทำตอนนี้ — ส่วน 2 (≈ 10 นาที)
> 1. ▶️ รันเซลล์ **2.1** (ฟังก์ชันและค่าคงที่) แล้วรันเซลล์ **2.2** (ดึงข้อมูล ใช้เวลา 1–3 นาที)
> 2. 👀 อ่านตาราง "ที่มาของข้อมูล" — ถ้าเป็น ⚠️ จำลอง ให้จดไว้ใน Prompt Log
> 3. ยังไม่ต้องเขียน Explain This ในส่วนนี้ (ส่วนถัดไปจะใช้ข้อมูลเหล่านี้)

In [ ]:
# 2.1 ค่าคงที่ + ตัวช่วยโหลดข้อมูล [โค้ดอ้างอิงจากอาจารย์]
import os, urllib.request, requests
from scipy.ndimage import gaussian_filter
from shapely.geometry import Point, box
from rasterio.enums import Resampling
from pyproj import Transformer

KK_LON, KK_LAT = 102.833, 16.432      # ใจกลางเมืองขอนแก่น (lon, lat เป็นองศา EPSG:4326)
HALF_M = 5000                         # ครึ่งหนึ่งของกรอบศึกษา (10 x 10 กม.)
dlat = HALF_M / 111_320
dlon = HALF_M / (111_320 * np.cos(np.radians(KK_LAT)))
STUDY_BBOX = (KK_LON - dlon, KK_LAT - dlat, KK_LON + dlon, KK_LAT + dlat)   # (west, south, east, north) องศา
PAD = 0.15                            # ขอบเผื่อเป็นองศา สำหรับชั้นที่หยาบกว่า
PAD_BBOX = (STUDY_BBOX[0]-PAD, STUDY_BBOX[1]-PAD, STUDY_BBOX[2]+PAD, STUDY_BBOX[3]+PAD)
UTM = "EPSG:32648"                    # UTM โซน 48N — ขอนแก่น (~102.8°E) อยู่ในโซน 102–108°E
SCALE, OFFSET = 0.0001, -0.1          # Sentinel-2 L2A: reflectance = DN*0.0001 - 0.1 (baseline >= 04.00)
BAND_NAMES = ["blue", "green", "red", "nir"]
DATA_SOURCE = {}                      # บันทึกว่าแต่ละชั้นมาจากของจริงหรือชุดจำลอง
rng = np.random.default_rng(42)

def _utm_grid(bbox, res=10):
    """สร้างพิกัดกริด UTM (กึ่งกลางพิกเซล) ครอบ bbox องศา ที่ความละเอียด res เมตร"""
    t = Transformer.from_crs("EPSG:4326", UTM, always_xy=True)
    l, b = t.transform(bbox[0], bbox[1]); r, tp = t.transform(bbox[2], bbox[3])
    l, b = np.floor(l/res)*res, np.floor(b/res)*res
    r, tp = np.ceil(r/res)*res, np.ceil(tp/res)*res
    x = np.arange(l + res/2, r, res); y = np.arange(tp - res/2, b, -res)
    return x, y

# ---------- ชุดจำลอง (ใช้เมื่อดึงของจริงไม่ได้) ----------
def _synthetic_districts():
    t = Transformer.from_crs("EPSG:4326", UTM, always_xy=True)
    cx, cy = t.transform(KK_LON, KK_LAT)
    polys, names = [], []
    k = 0
    for i in range(-1, 2):
        for j in range(-1, 1):
            k += 1
            x0, y0 = cx + i*9000 - 4500, cy + j*9000 - 3000
            polys.append(box(x0, y0, x0 + 9000, y0 + 9000)); names.append(f"อำเภอจำลอง {k}")
    return gpd.GeoDataFrame({"shapeName": names}, geometry=polys, crs=UTM).to_crs("EPSG:4326")

def _synthetic_cube(n_dates=3):
    x, y = _utm_grid(STUDY_BBOX)
    h, w = len(y), len(x)
    v = gaussian_filter(rng.normal(size=(h, w)), 25); v = (v - v.min()) / (v.max() - v.min())
    wat = gaussian_filter(rng.normal(size=(h, w)), 40); water = wat > np.percentile(wat, 94)
    times, arrs = pd.to_datetime(["2026-01-10", "2026-02-09", "2026-03-11"][:n_dates]), []
    for k in range(len(times)):
        vk = v * (1 - 0.25 * k / max(len(times) - 1, 1))
        red = 0.08 + 0.12 * (1 - vk); nir = 0.15 + 0.35 * vk
        green, blue = 0.07 + 0.05 * vk, 0.05 + 0.03 * (1 - vk)
        red[water], nir[water], green[water], blue[water] = 0.03, 0.02, 0.06, 0.05
        dn = np.stack([blue, green, red, nir]); dn = np.clip((dn - OFFSET) / SCALE, 0, 65535).astype("uint16")
        arrs.append(dn)
    da = xr.DataArray(np.stack(arrs), dims=("time", "band", "y", "x"),
                      coords=dict(time=times, band=BAND_NAMES, y=y, x=x), name="s2")
    return da.rio.write_crs(UTM)

def _synthetic_pop():
    res = 1/120   # ~0.00833 องศา ~ 1 กม.
    x = np.arange(PAD_BBOX[0], PAD_BBOX[2], res) + res/2; y = np.arange(PAD_BBOX[3], PAD_BBOX[1], -res) - res/2
    X, Y = np.meshgrid(x, y)
    d2 = ((X - KK_LON)/0.08)**2 + ((Y - KK_LAT)/0.08)**2
    pop = (3500*np.exp(-d2) + 120*rng.random(X.shape)).astype("float32")
    da = xr.DataArray(pop, dims=("y", "x"), coords=dict(y=y, x=x), name="pop_density")
    return da.rio.write_crs("EPSG:4326")

# ---------- ตัวโหลดของจริง ----------
def load_districts():
    try:
        meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/THA/ADM2/", timeout=30).json()
        meta = meta[0] if isinstance(meta, list) else meta
        url = meta.get("simplifiedGeometryGeoJSON") or meta["gjDownloadURL"]
        gdf = gpd.read_file(url)
        gdf = gdf[gdf.intersects(box(*PAD_BBOX))][["shapeName", "geometry"]].reset_index(drop=True)
        assert len(gdf) > 0, "ไม่พบอำเภอในกรอบ"
        DATA_SOURCE["อำเภอ"] = "✅ จริง: geoBoundaries gbOpen THA ADM2"
        return gdf
    except Exception as e:
        print("⚠️ โหลดขอบเขตอำเภอจริงไม่ได้ →", type(e).__name__, str(e)[:100])
        DATA_SOURCE["อำเภอ"] = "⚠️ ชุดจำลอง"
        return _synthetic_districts()

def load_s2_cube(n_dates=3, datetime="2026-01-01/2026-03-31", max_cloud=15):
    try:
        from pystac_client import Client
        cat = Client.open("https://earth-search.aws.element84.com/v1")
        items = list(cat.search(collections=["sentinel-2-l2a"], bbox=STUDY_BBOX, datetime=datetime,
                                query={"eo:cloud_cover": {"lt": max_cloud}}, max_items=60).items())
        assert items, "ค้นหาสำเร็จแต่ไม่พบภาพ"
        tile = lambda it: it.properties.get("grid:code") or it.properties.get("s2:mgrs_tile")
        best = min(items, key=lambda it: it.properties["eo:cloud_cover"])
        same = sorted([it for it in items if tile(it) == tile(best)], key=lambda it: it.datetime)
        picked, last = [], None                       # เลือกวันที่ห่างกัน >= 10 วัน
        for it in same:
            if last is None or (it.datetime - last).days >= 10:
                picked.append(it); last = it.datetime
        picked = picked[:n_dates]
        arrs = []
        for it in picked:
            bands = []
            for b in BAND_NAMES:
                a = rioxarray.open_rasterio(it.assets[b].href).squeeze("band", drop=True)
                bands.append(a.rio.clip_box(*STUDY_BBOX, crs="EPSG:4326").load())
            arrs.append(xr.concat(bands, dim=pd.Index(BAND_NAMES, name="band")))
        ref = arrs[0]
        arrs = [ref] + [a.rio.reproject_match(ref) for a in arrs[1:]]
        cube = xr.concat(arrs, dim=pd.Index([pd.Timestamp(it.datetime).tz_localize(None) for it in picked], name="time"))
        DATA_SOURCE["Sentinel-2"] = f"✅ จริง: Earth Search STAC ({len(picked)} วัน, tile {tile(best)})"
        return cube.rio.write_crs(ref.rio.crs)
    except Exception as e:
        print("⚠️ โหลด Sentinel-2 จริงไม่ได้ →", type(e).__name__, str(e)[:100])
        DATA_SOURCE["Sentinel-2"] = "⚠️ ชุดจำลอง"
        return _synthetic_cube(n_dates)

def load_pop():
    url = "https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km_UNadj/2020/THA/tha_pd_2020_1km_UNadj.tif"
    path = "tha_pd_2020_1km_UNadj.tif"     # ถ้าอาจารย์แจกไฟล์นี้ใน Classroom ให้อัปโหลดไว้ที่ /content แล้วเซลล์จะใช้ไฟล์นี้
    try:
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)
        a = rioxarray.open_rasterio(path, masked=True).squeeze("band", drop=True)
        a = a.rio.clip_box(*PAD_BBOX, crs="EPSG:4326").load()
        DATA_SOURCE["WorldPop"] = "✅ จริง: WorldPop 2020 1 กม. (persons/km²)"
        return a
    except Exception as e:
        print("⚠️ โหลด WorldPop จริงไม่ได้ →", type(e).__name__, str(e)[:100])
        DATA_SOURCE["WorldPop"] = "⚠️ ชุดจำลอง"
        return _synthetic_pop()

In [ ]:
# 2.2 ดึงข้อมูลทั้งสามชั้น (ใช้เวลา ~1–3 นาที ขึ้นกับอินเทอร์เน็ต)
amphoe = load_districts()
cube   = load_s2_cube()
pop    = load_pop()
print("\n== ที่มาของข้อมูล ==")
for k, v in DATA_SOURCE.items():
    print(f"{k:12s} {v}")
print("\ncube dims:", dict(cube.sizes))

## ส่วน 3 — เวกเตอร์: CRS, องศากับเมตร, และพื้นที่

> ### ✅ ทำตอนนี้ — ส่วน 3 (≈ 25 นาที)
> 1. ▶️ รันเซลล์ **3.1** → อ่านบัตรประจำตัวของ `amphoe` (หน่วยของ CRS คืออะไร)
> 2. ▶️ รันเซลล์ **3.2** → เดินไปทางตะวันออกกับทางเหนือ 0.1° ได้ระยะเท่ากันไหม
> 3. ▶️ รันเซลล์ **3.3** → เปรียบเทียบพื้นที่ที่ได้จาก EPSG:4326 กับ EPSG:32648 (เซลล์นี้จะมี warning ของ geopandas ให้อ่านด้วย)
> 4. 📝 เขียน **Explain This #2** ของเซลล์ 3.3 (ทำไมค่าในคอลัมน์ `area_deg2` ใช้ไม่ได้ และ `to_crs` ทำอะไรกับตัวเลขพิกัด)

In [ ]:
# 3.1 บัตรประจำตัวของขอบเขตอำเภอ
_ = check_layer(amphoe, "อำเภอ (ต้นฉบับ)")
amphoe.head()

In [ ]:
# 3.2 หนึ่งองศา ไม่เท่ากับระยะเท่ากันทุกทิศ
from pyproj import Geod
geod = Geod(ellps="WGS84")
_, _, d_east  = geod.inv(KK_LON, KK_LAT, KK_LON + 0.1, KK_LAT)
_, _, d_north = geod.inv(KK_LON, KK_LAT, KK_LON, KK_LAT + 0.1)
print(f"จากขอนแก่น เดินไปทางตะวันออก 0.1° = {d_east/1000:.2f} กม.")
print(f"จากขอนแก่น เดินไปทางเหนือ    0.1° = {d_north/1000:.2f} กม.")

In [ ]:
# 3.3 พื้นที่: ผิดวิธี vs ถูกวิธี
amphoe["area_deg2"] = amphoe.geometry.area                    # ใน EPSG:4326 หน่วยคือ "ตารางองศา"
amphoe_utm = amphoe.to_crs("EPSG:32648")                      # แปลงพิกัดเป็นเมตร (UTM 48N)
amphoe_utm["area_km2"] = amphoe_utm.geometry.area / 1e6       # ตารางเมตร -> ตารางกิโลเมตร
print(pd.concat([amphoe[["shapeName", "area_deg2"]], amphoe_utm[["area_km2"]]], axis=1).round(4).to_string(index=False))
_ = check_layer(amphoe_utm, "อำเภอ (UTM 48N)")

### Explain This #2 — เซลล์ 3.3
- **ทำอะไร:** _(เขียนเอง)_
- **ข้อมูลเปลี่ยนอย่างไร:** _(ตัวเลขพิกัดเปลี่ยนไหม? หน่วยเปลี่ยนจากอะไรเป็นอะไร?)_
- **จุดเสี่ยง + วิธีตรวจ:** _(ถ้าไม่ใช้ `to_crs` จะผิดอย่างไร และคุณจะตรวจว่าพื้นที่ "สมเหตุสมผล" ได้อย่างไร)_

In [ ]:
log_prompt(
    prompt="(ถ้าใช้ AI ช่วยอธิบาย/แก้โค้ดส่วนที่ 3 ให้วาง prompt ที่ใช้ ถ้าไม่ได้ใช้ ให้ลบเซลล์นี้ทิ้ง)",
    result="สรุปสั้น ๆ ว่า AI ตอบอะไร",
    verdict="แก้",
    note="สิ่งที่คุณตรวจ/แก้เอง",
)

## ส่วน 4 — ราสเตอร์: อ่าน "บัตรประจำตัว" แล้วทำให้สามชั้นอยู่บนกริดเดียวกัน

> ### ✅ ทำตอนนี้ — ส่วน 4 (≈ 30 นาที)
> 1. ▶️ รันเซลล์ **4.1** และ **4.2** → อ่านค่า `transform`, ความละเอียด, ขอบเขตของภาพ Sentinel-2 และดูภาพสีจริงที่แกนเป็นเมตร
> 2. ▶️ รันเซลล์ **4.3** → ตารางเทียบสามชั้น 📝 เขียน **Explain This #3**: ความต่างข้อใดบ้างที่ทำให้ซ้อนชั้นไม่ได้ และต้องแก้ด้วยอะไร
> 3. 🤖 **ภารกิจ Prompt #1** (ด้านล่าง) ลองให้ AI เขียน reproject ก่อนรันเซลล์อ้างอิง 4.4
> 4. ▶️ รัน 4.4 → 👀 เทียบภาพ nearest กับ bilinear 📝 เขียน **Explain This #4**
> 5. ▶️ รัน 4.5 (สร้างแผนที่ประเภทพื้นที่ `lulc` ง่าย ๆ ไว้ใช้ใน Bug Hunt)

In [ ]:
# 4.1 บัตรประจำตัวของภาพ Sentinel-2 (วันแรก แบนด์แดง) และ WorldPop
t0 = cube.isel(time=0)                       # (band, y, x) ของวันแรก
s2_red = t0.sel(band="red")                  # (y, x) ใช้เป็น "กริดอ้างอิง" ต่อไป
_ = check_layer(s2_red, "Sentinel-2 (แดง)")
_ = check_layer(pop, "WorldPop")
print("\ntransform (affine):", s2_red.rio.transform())
print("พิกเซลมุมซ้ายบน (กึ่งกลาง) x, y =", float(s2_red.x[0]), float(s2_red.y[0]), "| nodata:", s2_red.rio.nodata, "| dtype:", s2_red.dtype)

In [ ]:
# 4.2 ภาพสีจริง — ดูแกนว่าเป็นหน่วยอะไร
def reflectance(da):
    return da.astype("float32") * SCALE + OFFSET        # DN -> reflectance (0–1)

def show_rgb(t, ax, title=""):
    rgb = reflectance(t.sel(band=["red", "green", "blue"]))
    img = np.clip(np.moveaxis(rgb.values, 0, -1) / 0.3, 0, 1)
    l, b, r, tp = t.rio.bounds()
    ax.imshow(img, extent=(l, r, b, tp)); ax.set_title(title)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")

fig, ax = plt.subplots(figsize=(6, 6))
show_rgb(t0, ax, f"Sentinel-2 {str(t0.time.values)[:10]}  ({s2_red.rio.crs})")
plt.show()

In [ ]:
# 4.3 ตารางเทียบสามชั้น: ซ้อนกันได้ไหม?
_ = check_layer(amphoe, "อำเภอ", quiet=True)
tbl = pd.DataFrame([r for r in LAYER_ROWS if r["layer"] in ("อำเภอ", "Sentinel-2 (แดง)", "WorldPop")])
print(tbl[["layer", "kind", "crs", "unit", "res"]].drop_duplicates("layer").to_string(index=False))
print("\nขอบเขต (ซ้าย, ล่าง, ขวา, บน):")
for r in tbl.drop_duplicates("layer").itertuples():
    print(f"  {r.layer:18s} {r.bounds}")

### Explain This #3 — ตารางเทียบสามชั้น (เซลล์ 4.3)
- **ทำอะไร:** _(เขียนเอง)_
- **ข้อมูลเปลี่ยนอย่างไร:** _(ชั้นไหนอยู่ในองศา ชั้นไหนอยู่ในเมตร ความละเอียดต่างกันเท่าไร)_
- **ถ้าซ้อนทั้งสามชั้นโดยไม่แก้อะไร จะเกิดอะไร และต้องทำอะไรบ้าง (ตามลำดับ):** _..._

### ภารกิจ Prompt #1 — ให้ AI เขียนโค้ด "ย้าย WorldPop ไปอยู่บนกริดเดียวกับภาพ Sentinel-2"
ใช้เทมเพลตของวิชา (`docs/prompt-template.md`) และ **ต้องระบุให้ครบ**: CRS ต้นทางและปลายทาง · หน่วย · ความละเอียดเป้าหมาย · ชื่อตัวแปรเข้า/ออก · **วิธี resample พร้อมเหตุผล** · ไลบรารี (`rioxarray`)

ข้อมูลเข้าให้บอก AI: `pop` = `xarray.DataArray` (y, x) จาก `rioxarray`, EPSG:4326, ~0.0083°/พิกเซล, ค่าเป็น persons/km² · `s2_red` = `DataArray` (y, x) EPSG:32648, 10 ม. · ผลลัพธ์ที่ต้องการ `pop_bil` อยู่บนกริดเดียวกับ `s2_red`

> ⚠️ **ห้ามรันเซลล์อ้างอิง 4.4 ก่อนลอง prompt อย่างน้อย 1 รอบ** — แล้วเทียบโค้ดของ AI กับของอาจารย์ (ถ้าไม่ตรงกัน ใครถูก? พิสูจน์ด้วย `check_layer`)

In [ ]:
log_prompt(
    prompt="(วาง prompt ทั้งหมดที่ส่งให้ AI)",
    result="อธิบายผลของโค้ดที่รันได้ (CRS/ความละเอียด/ขอบเขตหลัง reproject ตรงกับ s2_red ไหม)",
    verdict="แก้",
    note="สิ่งที่คุณตรวจ/แก้เอง เช่น เพิ่ม check_layer ตรวจหลังรัน",
)

In [ ]:
# 4.4 [โค้ดอ้างอิงจากอาจารย์] ย้าย WorldPop ไปกริดเดียวกับ Sentinel-2 ด้วยสองวิธี resample
ref = s2_red                                                     # กริดเป้าหมาย (UTM 48N, 10 ม.)
pop_near = pop.rio.reproject_match(ref, resampling=Resampling.nearest)
pop_bil  = pop.rio.reproject_match(ref, resampling=Resampling.bilinear)
_ = check_layer(pop_bil, "WorldPop -> กริด S2")

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, da, name in zip(axs, (pop_near, pop_bil), ("nearest", "bilinear")):
    da.plot(ax=ax, cmap="magma", add_colorbar=False)
    ax.set_title(f"WorldPop เป็น 10 ม. ({name})"); ax.set_aspect("equal")
plt.show()
u = lambda a: len(np.unique(a.values[~np.isnan(a.values)]))
print("จำนวนค่าไม่ซ้ำ  nearest:", u(pop_near), "| bilinear:", u(pop_bil))
print("ค่าเฉลี่ย        nearest:", round(float(pop_near.mean()), 1), "| bilinear:", round(float(pop_bil.mean()), 1))

### Explain This #4 — เซลล์ 4.4 `reproject_match`
- **ทำอะไร:** _(เขียนเอง)_
- **ข้อมูลเปลี่ยนอย่างไร:** _(CRS, ความละเอียด, ขนาดภาพเปลี่ยนจากอะไรเป็นอะไร และภาพ bilinear "คมขึ้น" จริงไหม — ดูจากตัวเลขที่พิมพ์)_
- **ถ้าชั้นนี้เป็น "แผนที่ประเภทที่ดิน" (รหัส 1, 2, 3, 4) จะเลือก resampling อะไร และทำไม:** _..._

In [ ]:
# 4.5 สร้างแผนที่ประเภทพื้นที่ง่าย ๆ (lulc: 1=น้ำ 2=ดินโล่ง/สิ่งปลูกสร้าง 3=พืชโปร่ง/นา 4=พืชหนาแน่น) จากดัชนีของวันแรก
red, grn, nir = (reflectance(t0.sel(band=b)) for b in ("red", "green", "nir"))
ndvi0 = ((nir - red) / (nir + red)).rio.write_crs(UTM)
ndwi0 = ((grn - nir) / (grn + nir)).rio.write_crs(UTM)
lulc = xr.where(ndwi0 > 0.1, 1, xr.where(ndvi0 < 0.2, 2, xr.where(ndvi0 < 0.5, 3, 4))).astype("float32")
lulc = lulc.rio.write_crs(UTM)
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
ndvi0.plot(ax=axs[0], cmap="RdYlGn", vmin=-0.2, vmax=0.9); axs[0].set_title("NDVI วันแรก"); axs[0].set_aspect("equal")
lulc.plot(ax=axs[1], cmap="viridis", levels=[0.5, 1.5, 2.5, 3.5, 4.5]); axs[1].set_title("lulc 1–4"); axs[1].set_aspect("equal")
plt.show()
print("รหัสที่พบใน lulc:", sorted(np.unique(lulc.values).tolist()))

## ส่วน 5 — Bug Hunt: โค้ดที่ "รันผ่าน" แต่ผิดเชิงพื้นที่ 4 จุด

โค้ดสองบล็อกด้านล่าง **AI เขียนให้** และรันผ่านโดยไม่มี error สีแดง — แต่มีบั๊กเชิงพื้นที่ซ่อนอยู่ **4 จุด** (บล็อก A มี 3 จุด, บล็อก B มี 1 จุด) งานของคุณคือ **ชี้บรรทัด** อธิบายว่าผิดเพราะอะไร แก้ และพิสูจน์ว่าแก้ถูก

> ### ✅ ทำตอนนี้ — ส่วน 5 (≈ 40 นาที)
> 1. 📝 **อ่านโค้ดก่อนรัน** ทีละบรรทัด เขียนในตารางบั๊กด้านล่างว่าสงสัยบรรทัดไหนและเพราะอะไร *(ก่อนเห็นผลลัพธ์)*
> 2. ▶️ รันบล็อก A และ B → 👀 ดูตัวเลขที่พิมพ์ออกมา: มีตัวเลขไหนที่ "ดูดี" แต่ตำแหน่งผิด?
> 3. ✏️ เขียนโค้ดที่แก้แล้วในเซลล์ 5.3 ตรวจด้วย **ตัวเลขอ้างอิง** ที่ให้ไว้ แล้ว 📝 กรอกตารางบั๊กและ **Explain This #5**
> 4. 🤖 ถ้าให้ AI ช่วยแก้ ให้ `log_prompt(...)` ทุกรอบ แล้วระบุว่าคุณตรวจผลอย่างไร
>
> **ตัวเลขอ้างอิงเพื่อตรวจคำตอบ (ไม่ใช่เฉลย):** พื้นที่วงกลมรัศมี 500 ม. ≈ 0.785 กม.² · ระยะ มข. – บึงแก่นนคร ≈ 6 กม. (ไม่ใช่ไม่กี่เมตร) · อย่างน้อย 1 อำเภอต้องอยู่ในกรอบภาพ · แผนที่ประเภทพื้นที่ต้องมีเฉพาะรหัสจำนวนเต็ม 1–4

In [ ]:
# 5.1 บล็อก A — โค้ดที่ "AI เขียนให้"
# Prompt ที่ส่งให้ AI: "สร้างจุดสถานี 3 แห่ง วัดระยะห่างและพื้นที่รัศมี 500 ม. แล้วตัดอำเภอให้เหลือเฉพาะส่วนที่อยู่ในภาพ Sentinel-2"
stations = pd.DataFrame({"name": ["มข.", "บึงแก่นนคร", "เขื่อนอุบลรัตน์"],
                         "lat": [16.472, 16.420, 16.775], "lon": [102.823, 102.833, 102.620]})
pts = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.lat, stations.lon), crs="EPSG:4326")   # บรรทัด A1
pts_m = pts.set_crs("EPSG:32648", allow_override=True)                                                     # บรรทัด A2
buf = pts_m.buffer(500)
print("พื้นที่รัศมี 500 ม. (กม.²):", (buf.area / 1e6).round(3).tolist())
d = pts_m.geometry.iloc[0].distance(pts_m.geometry.iloc[1])
print("ระยะ มข. – บึงแก่นนคร (ม.):", round(d, 1))
img_box = box(*s2_red.rio.bounds())                      # กรอบของภาพ Sentinel-2 (UTM 48N)
in_img = gpd.clip(amphoe, img_box)                       # บรรทัด A3
print("อำเภอที่อยู่ในภาพ:", len(in_img))

In [ ]:
# 5.2 บล็อก B — โค้ดที่ "AI เขียนให้"
# Prompt ที่ส่งให้ AI: "ลดความละเอียดแผนที่ประเภทพื้นที่ lulc จาก 10 ม. เป็น 100 ม. เพื่อทำแผนที่ภาพรวม"
lulc_100 = lulc.rio.reproject(lulc.rio.crs, resolution=100, resampling=Resampling.bilinear)   # บรรทัด B4
vals = lulc_100.values[~np.isnan(lulc_100.values)]
print("จำนวนเซลล์ 100 ม.:", vals.size)
print("ตัวอย่างรหัสประเภทที่พบ:", sorted(set(np.round(vals, 1).tolist()))[:10])

### ตารางบั๊ก (กรอกเอง)
| # | บรรทัด | ผิดเพราะอะไร (เชิงพื้นที่) | แก้เป็น | พิสูจน์อย่างไรว่าแก้ถูก |
|---|---|---|---|---|
| 1 | _..._ | _..._ | _..._ | _..._ |
| 2 | _..._ | _..._ | _..._ | _..._ |
| 3 | _..._ | _..._ | _..._ | _..._ |
| 4 | _..._ | _..._ | _..._ | _..._ |

### Explain This #5 — Bug Hunt
- **บั๊กไหนที่ตัวเลขดูถูกแต่ตำแหน่งผิด และทำไมการตรวจแค่ตัวเลขจึงไม่พอ:** _..._
- **วิธีตรวจที่คุณจะใช้ทุกครั้งหลังรันโค้ดที่ AI เขียน (อย่างน้อย 3 ข้อ):** _..._

In [ ]:
# 5.3 ✏️ เขียนโค้ดที่แก้แล้วของคุณที่นี่ (บล็อก A และ B) แล้วพิมพ์ตัวเลขเทียบกับ "ตัวเลขอ้างอิง"
# เริ่มจากคัดลอกบล็อก A/B มาแก้ทีละบรรทัด

## ส่วน 6 — Clip และ Rasterize

> ### ✅ ทำตอนนี้ — ส่วน 6 (≈ 30 นาที)
> 1. ▶️ รันเซลล์ **6.1** → ดู error ที่เกิดจากการ clip ด้วยเวกเตอร์ที่ CRS ไม่ตรง แล้วดูวิธีแก้และภาพ NDVI ที่ตัดตามขอบเขตอำเภอ
> 2. ▶️ รันเซลล์ **6.2** → rasterize ขอบเขตอำเภอลงกริด 10 ม. แล้วหา NDVI เฉลี่ยรายอำเภอ
> 3. 📝 เขียน **Explain This #6** (ทำไม clip ครั้งแรกล้มเหลว · rasterize แปลงอะไรเป็นอะไร · `fill=0` หมายถึงอะไร)
> 4. 👀 ตรวจ: ผลรวมพื้นที่รายอำเภอควร ≈ 100 กม.² (กรอบศึกษา 10 × 10 กม.) ถ้าห่างมาก ให้บอกว่าเป็นเพราะอะไร

In [ ]:
# 6.1 clip ราสเตอร์ด้วยขอบเขตอำเภอ — ทำพลาดก่อน แล้วค่อยแก้
mueang = amphoe[amphoe.contains(Point(KK_LON, KK_LAT))]          # อำเภอที่มีจุดกลางขอนแก่นอยู่ข้างใน
print("อำเภอที่เลือก:", mueang.shapeName.tolist(), "| CRS ของเวกเตอร์:", mueang.crs)
try:
    ndvi0.rio.clip(mueang.geometry, drop=True)                   # ผิด: เวกเตอร์เป็นองศา แต่ภาพเป็นเมตร
except Exception as e:
    print("❌", type(e).__name__, ":", str(e)[:140])

mueang_utm = mueang.to_crs(ndvi0.rio.crs)                        # แก้: แปลงเวกเตอร์ให้ CRS ตรงกับราสเตอร์ก่อน
ndvi_mueang = ndvi0.rio.clip(mueang_utm.geometry, drop=True)
print("หลัง clip shape:", ndvi_mueang.shape, "| พิกเซลที่ไม่ใช่ NaN:", int(ndvi_mueang.notnull().sum()))
fig, ax = plt.subplots(figsize=(6, 6))
ndvi_mueang.plot(ax=ax, cmap="RdYlGn", vmin=-0.2, vmax=0.9); mueang_utm.boundary.plot(ax=ax, color="black")
ax.set_aspect("equal"); plt.show()

In [ ]:
# 6.2 rasterize: เปลี่ยนโพลิกอนอำเภอเป็นกริด 10 ม. (ค่าในแต่ละพิกเซล = หมายเลขอำเภอ) แล้วหา NDVI เฉลี่ยรายอำเภอ
from rasterio import features
amphoe_utm = amphoe.to_crs(ndvi0.rio.crs).reset_index(drop=True)
shapes = ((geom, i + 1) for i, geom in enumerate(amphoe_utm.geometry))
zone = features.rasterize(shapes, out_shape=ndvi0.shape, transform=ndvi0.rio.transform(), fill=0, dtype="int16")

rows = []
for i, nm in enumerate(amphoe_utm.shapeName, start=1):
    m = zone == i
    if m.sum() == 0:
        continue
    rows.append({"อำเภอ": nm, "พิกเซล": int(m.sum()), "พื้นที่_กม2": round(m.sum() * 100 / 1e6, 2),
                 "NDVI_เฉลี่ย": round(float(np.nanmean(ndvi0.values[m])), 3)})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("\nผลรวมพื้นที่ (กม.²):", round(summary["พื้นที่_กม2"].sum(), 2), "| พิกเซลที่ไม่ตกอำเภอใดเลย (zone=0):", int((zone == 0).sum()))

### Explain This #6 — clip และ rasterize
- **ทำอะไร:** _(เขียนเอง)_
- **ข้อมูลเปลี่ยนอย่างไร:** _(เวกเตอร์เปลี่ยนเป็นอะไร ตัวเลขในกริด `zone` หมายถึงอะไร)_
- **จุดเสี่ยง + วิธีตรวจ:** _(ถ้าไม่ `to_crs` ก่อน rasterize จะเกิดอะไร · พื้นที่ "ไม่ตกอำเภอใด" คืออะไร)_

## ส่วน 7 — Data Cube: เวลา × แบนด์ × y × x

> ### ✅ ทำตอนนี้ — ส่วน 7 (≈ 15 นาที)
> 1. ▶️ รัน **7.1** → อ่านรูปทรงของ `cube` และ `ndvi_cube` (ชื่อมิติ ไม่ใช่แค่ตัวเลข)
> 2. 👀 ดูกราฟ NDVI เฉลี่ยตามเวลา และแผนที่ผลต่าง NDVI วันสุดท้ายเทียบวันแรก
> 3. 📝 เขียน **Explain This #7** (ทำไมการเลือก `.sel(band="red")` ปลอดภัยกว่า `cube[2]` · `dim=("y","x")` ยุบมิติอะไร)

In [ ]:
# 7.1 สร้างดัชนีทั้งก้อน cube แล้วสรุปตามเวลา
print(cube)
red_c, nir_c = reflectance(cube.sel(band="red")), reflectance(cube.sel(band="nir"))
ndvi_cube = ((nir_c - red_c) / (nir_c + red_c)).rio.write_crs(UTM)       # (time, y, x)
print("\nndvi_cube มิติ:", dict(ndvi_cube.sizes))

mean_all = ndvi_cube.mean(dim=("y", "x"))
mean_m = ndvi_cube.rio.clip(mueang_utm.geometry, drop=True).mean(dim=("y", "x"))
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(ndvi_cube.time, mean_all, "o-", label="ทั้งกรอบ"); axs[0].plot(ndvi_cube.time, mean_m, "s--", label=f"อำเภอ {mueang.shapeName.iloc[0]}")
axs[0].set_title("NDVI เฉลี่ยตามเวลา"); axs[0].legend(); axs[0].tick_params(axis="x", rotation=30)
if ndvi_cube.sizes["time"] > 1:
    diff = ndvi_cube.isel(time=-1) - ndvi_cube.isel(time=0)
    diff.plot(ax=axs[1], cmap="RdBu", center=0); axs[1].set_title("NDVI วันสุดท้าย − วันแรก"); axs[1].set_aspect("equal")
plt.tight_layout(); plt.show()

### Explain This #7 — data cube
- **ทำอะไร:** _(เขียนเอง)_
- **ข้อมูลเปลี่ยนอย่างไร:** _(มิติของ `cube` กับ `ndvi_cube` ต่างกันอย่างไร ทำไม)_
- **จุดเสี่ยง + วิธีตรวจ:** _(ถ้าสองวันมีกริดไม่ตรงกันจะเกิดอะไร — โค้ดตอนโหลดแก้ด้วยอะไร)_

## ส่วน 8 — Mission 2 และ Verify on map

**ภารกิจ Prompt #2:** ขอให้ AI ประมาณ **จำนวนประชากรโดยประมาณในรัศมี 2 กม. รอบใจกลางเมืองขอนแก่น** จากชั้น WorldPop (`pop`, persons/km²)
ต้องระบุใน prompt: จุดศูนย์กลางเป็น lon/lat องศา · บัฟเฟอร์ต้องทำใน CRS ที่หน่วยเป็นเมตร (ระบุ EPSG) · ชื่อตัวแปรเข้า/ออก · วิธีตรวจผลด้วยวิธีที่สอง

> ### ✅ ทำตอนนี้ — ส่วน 8 (≈ 20 นาที)
> 1. 🤖 ส่ง Prompt #2 ให้ AI ลองรันโค้ดที่ได้ (ถ้าไม่ผ่านใน 2 รอบ ใช้เซลล์อ้างอิง 8.2) แล้ว `log_prompt(...)`
> 2. ▶️ รัน **8.2** → ดูการประมาณ 2 วิธี (A: บนกริด UTM 10 ม., B: บนราสเตอร์ต้นฉบับ) ต้องได้ค่าใกล้กัน
> 3. ▶️ รัน **8.3** (แผนที่ Verify on map) 👀 ตรวจว่าทุกชั้นซ้อนกันตรงตำแหน่ง
> 4. 📝 กรอก **เช็กลิสต์ CRS** ด้านล่าง

In [ ]:
log_prompt(
    prompt="(วาง Prompt #2 ทั้งหมด)",
    result="สรุปผลที่ได้ (ตัวเลขประชากร + พื้นที่บัฟเฟอร์ที่คำนวณได้)",
    verdict="แก้",
    note="ตรวจด้วยวิธีที่สองอย่างไร ผลต่างเท่าไร",
)

In [ ]:
# 8.2 [โค้ดอ้างอิงจากอาจารย์] ประชากรโดยประมาณในรัศมี 2 กม. รอบใจกลางขอนแก่น
to_utm = Transformer.from_crs("EPSG:4326", UTM, always_xy=True)       # always_xy=True: ลำดับ lon, lat
x0, y0 = to_utm.transform(KK_LON, KK_LAT)
buf2k = gpd.GeoDataFrame(geometry=[Point(x0, y0).buffer(2000)], crs=UTM)       # บัฟเฟอร์ใน "เมตร"
area_km2 = buf2k.area.iloc[0] / 1e6
print(f"พื้นที่บัฟเฟอร์ = {area_km2:.3f} กม.² (ควร ≈ 12.566)")

popA = pop_bil.rio.clip(buf2k.geometry, drop=True)                              # วิธี A: กริด UTM 10 ม.
estA = float(popA.mean()) * area_km2
popB = pop.rio.clip(buf2k.to_crs(pop.rio.crs).geometry, drop=True)               # วิธี B: ราสเตอร์ต้นฉบับ (องศา)
estB = float(popB.mean()) * area_km2
print(f"ประชากรโดยประมาณ วิธี A: {estA:,.0f} คน | วิธี B: {estB:,.0f} คน | ต่างกัน {abs(estA-estB)/estB:.1%}")
print("(ความหนาแน่นเฉลี่ย × พื้นที่ เป็นการประมาณหยาบ เพราะพิกเซล WorldPop 1 กม. ใหญ่เทียบกับวงกลม 2 กม.)")

In [ ]:
# 8.3 Verify on map: ซ้อนภาพ ขอบเขตอำเภอ และบัฟเฟอร์ — ทุกชั้นต้องอยู่ใน CRS เดียวกันก่อนวาด
for g, nm in ((amphoe_utm, "อำเภอ (UTM)"), (buf2k, "บัฟเฟอร์ 2 กม. (UTM)")):
    check_layer(g, nm, quiet=True)
fig, ax = plt.subplots(figsize=(7, 7))
show_rgb(t0, ax, "Verify on map: ภาพ + อำเภอ + บัฟเฟอร์ 2 กม.")
amphoe_utm.boundary.plot(ax=ax, color="yellow", linewidth=1.5)
buf2k.boundary.plot(ax=ax, color="red", linewidth=2)
ax.plot(x0, y0, "r*", ms=14)
l, b, r, tp = t0.rio.bounds(); ax.set_xlim(l, r); ax.set_ylim(b, tp)
plt.show()

### เช็กลิสต์ CRS — 5 ข้อที่ต้องตอบได้ก่อนเชื่อผลลัพธ์ทุกครั้ง (กรอกจากงานของคุณวันนี้)
- [ ] **CRS ของทุกชั้นตรงกันไหม** — ตรวจด้วย: _..._
- [ ] **หน่วยของ CRS คืออะไร** (องศา/เมตร) ก่อน buffer/area/distance — ตรวจด้วย: _..._
- [ ] **ความละเอียดและขอบเขต** ของสองชั้นที่จะซ้อนกัน — ตรวจด้วย: _..._
- [ ] **resample ถูกวิธีไหม** (หมวดหมู่ = nearest · ค่าต่อเนื่อง = bilinear/average) — ตรวจด้วย: _..._
- [ ] **ดูบนแผนที่แล้ว** ว่าตำแหน่งตรงของจริง (ไม่เชื่อตัวเลขอย่างเดียว) — ตรวจด้วย: _..._

## ส่วน 9 — สรุป, Prompt Log และส่งงานใน Google Classroom

### คำถามสะท้อนคิด (ตอบสั้น ๆ 2–3 ประโยคต่อข้อ เชื่อมกับสิ่งที่เจอจริงวันนี้)
1. บั๊กไหนใน Bug Hunt ที่คุณจะไม่เจอเลยถ้าดูแค่ตัวเลขที่พิมพ์ออกมา?
2. ถ้าต้องใช้ข้อมูลจากสองหน่วยงานที่ใช้ CRS คนละแบบ คุณจะทำอะไรเป็นขั้นแรก (ก่อนเขียนโค้ดวิเคราะห์)?
3. ข้อมูลของคุณสัปดาห์หน้า (feature engineering) จะอาศัยสิ่งที่เรียนวันนี้อย่างไร?

**คำตอบของฉัน:**
1. _..._
2. _..._
3. _..._

> ### ✅ ทำตอนนี้ — ส่วน 9 (≈ 20 นาที)
> 1. 📝 ตอบคำถามสะท้อนคิดข้างบน
> 2. ▶️ รันเซลล์ **9.1** → ควรเห็น `บันทึกไฟล์: prompt_log_week02.md` (มี prompt อย่างน้อย 2 รายการ)
> 3. **ดาวน์โหลด Prompt Log:** ไอคอนโฟลเดอร์ 📁 ซ้ายมือ → `prompt_log_week02.md` → จุดสามจุด → **Download**
> 4. **ดาวน์โหลด notebook:** **File → Download → Download .ipynb** แล้วตั้งชื่อ `week02_lab_<รหัส>.ipynb`
> 5. เปิด **Google Classroom** → งาน **"แล็บสัปดาห์ 2"** → แนบไฟล์ .ipynb และ Prompt Log → **Turn in / ส่งงาน**
>
> ⚠️ ไฟล์ใน `/content` หายเมื่อปิด Colab — ดาวน์โหลดก่อนปิดหน้าต่างทุกครั้ง

In [ ]:
# 9.1 ส่งออก Prompt Log + บันทึกที่มาของข้อมูล (จริง/จำลอง) ไว้ท้ายไฟล์
save_prompt_log("prompt_log_week02.md")
with open("prompt_log_week02.md", "a", encoding="utf-8") as f:
    f.write("\n## ที่มาของข้อมูลในแล็บนี้\n" + "\n".join(f"- {k}: {v}" for k, v in DATA_SOURCE.items()) + "\n")
print(open("prompt_log_week02.md", encoding="utf-8").read()[:1500])

### เช็กลิสต์ส่งงาน
- [ ] ทุกส่วนมี *Explain This* ที่เขียนเอง ครบ 7 จุด (#1–#7) ไม่ใช่คัดลอกจาก AI
- [ ] ตารางบั๊ก (ส่วน 5) กรอกครบ 4 บรรทัด และมีโค้ดที่แก้แล้วในเซลล์ 5.3
- [ ] เช็กลิสต์ CRS (ส่วน 8) กรอกครบ 5 ข้อ
- [ ] `log_prompt` ทุกครั้งที่ใช้ AI และแนบ `prompt_log_week02.md`
- [ ] ส่งใน Google Classroom แล้วสถานะขึ้นว่า "Turned in"